In [0]:
CREATE TABLE DEPT (
    DEPTNO INT,
    DNAME VARCHAR(50),
    LOC VARCHAR(50)
);

INSERT INTO DEPT VALUES
(10, 'ACCOUNTING', 'NEW YORK'),
(20, 'RESEARCH', 'DALLAS'),
(30, 'SALES', 'CHICAGO'),
(40, 'OPERATIONS', 'BOSTON');

In [0]:
CREATE TABLE EMP (
    EMPNO INT,
    ENAME VARCHAR(50),
    JOB VARCHAR(50),
    MGR INT,
    HIREDATE DATE,
    SAL DECIMAL(10,2),
    COMM DECIMAL(10,2),
    DEPTNO INT
);

INSERT INTO EMP VALUES
(7369, 'SMITH', 'CLERK', 7902, '1980-12-17', 800, NULL, 20),
(7499, 'ALLEN', 'SALESMAN', 7698, '1981-02-20', 1600, 300, 30),
(7521, 'WARD', 'SALESMAN', 7698, '1981-02-22', 1250, 500, 30),
(7566, 'JONES', 'MANAGER', 7839, '1981-04-02', 2975, NULL, 20),
(7654, 'MARTIN', 'SALESMAN', 7698, '1981-09-28', 1250, 1400, 30),
(7698, 'BLAKE', 'MANAGER', 7839, '1981-05-01', 2850, NULL, 30),
(7782, 'CLARK', 'MANAGER', 7839, '1981-06-09', 2450, NULL, 10),
(7788, 'SCOTT', 'ANALYST', 7566, '1987-04-19', 3000, NULL, 20),
(7839, 'KING', 'PRESIDENT', NULL, '1981-11-17', 5000, NULL, 10),
(7844, 'TURNER', 'SALESMAN', 7698, '1981-09-08', 1500, 0, 30),
(7876, 'ADAMS', 'CLERK', 7788, '1987-05-23', 1100, NULL, 20),
(7900, 'JAMES', 'CLERK', 7698, '1981-12-03', 950, NULL, 30),
(7902, 'FORD', 'ANALYST', 7566, '1981-12-03', 3000, NULL, 20),
(7934, 'MILLER', 'CLERK', 7782, '1982-01-23', 1300, NULL, 10);

In [0]:
SELECT *
FROM EMP
WHERE SAL >
(
    SELECT AVG(SAL)
    FROM EMP
);

In [0]:
SELECT *
FROM EMP
WHERE DEPTNO IN
(
    SELECT DEPTNO
    FROM DEPT
    WHERE DNAME IN ('RESEARCH','SALES')
);

In [0]:
SELECT EMPNO,
       ENAME,
       SAL,
       CASE
           WHEN SAL <= 1000 THEN 1
           WHEN SAL <= 2000 THEN 2
           WHEN SAL <= 3000 THEN 3
           ELSE 4
       END AS SALARY_STATUS
FROM EMP;

In [0]:
-- This query uses a CORRELATED SUBQUERY to find all employees whose salary
-- is GREATER THAN the average salary of their OWN department.
--
-- Breakdown:
--   E1 is the outer query's EMP table (the "main" row being evaluated).
--   E2 is the inner query's EMP table (used to compute the department average).
--
-- For EACH row E1 in EMP:
--   1. The subquery computes AVG(SAL) for only the rows in E2 that share
--      the same DEPTNO as E1 (E1.DEPTNO = E2.DEPTNO).
--   2. If E1.SAL > that department-average, the row is returned.
--
-- Because the subquery references E1.DEPTNO (a column from the outer query),
-- it is re-executed for every row of the outer query — this is why it is
-- called a "correlated" subquery.
--
-- Result: employees earning above their department's average salary.

SELECT *
FROM EMP E1
WHERE SAL >
(
    SELECT AVG(SAL)
    FROM EMP E2
    WHERE E1.DEPTNO = E2.DEPTNO
);

In [0]:
SELECT
ENAME,
SAL,
(
    SELECT AVG(SAL)
    FROM EMP
) AS AVG_SALARY
FROM EMP;

In [0]:
-- Find the SECOND HIGHEST salary using a subquery in the FROM clause.
--
-- Breakdown:
--   1. Inner query (the FROM-clause subquery):
--      SELECT DISTINCT SAL FROM EMP ORDER BY SAL DESC
--      -> produces a list of unique salaries sorted from highest to lowest.
--   2. The LIMIT 1 OFFSET 1 clause:
--      -> skips the first row (highest salary) and returns the second row.
--   3. The outer query simply selects from that result so the value
--      can be used like a regular column.

SELECT *
FROM (
    SELECT DISTINCT SAL
    FROM EMP
    ORDER BY SAL DESC
    LIMIT 1 OFFSET 1
);

-- Alternative: correlated subquery approach (no FROM-clause subquery)
-- Uses a WHERE-clause subquery that COUNTs how many DISTINCT salaries
-- are strictly greater than each salary. If exactly 1 salary is higher,
-- this salary is the second highest.

SELECT *
FROM EMP
WHERE SAL = (
    SELECT DISTINCT SAL
    FROM EMP E1
    WHERE 1 = (
        SELECT COUNT(DISTINCT SAL)
        FROM EMP E2
        WHERE E2.SAL > E1.SAL
    )
);

-- EXPECTED RESULT (both queries): 3000  (shared by SCOTT and FORD)

In [0]:
-- Find the SECOND HIGHEST salary using the DENSE_RANK() window function.
--
-- Breakdown:
--   1. DENSE_RANK() OVER (ORDER BY SAL DESC) assigns a rank to each salary:
--        - 5000 -> rank 1 (highest)
--        - 3000 -> rank 2 (second highest)
--        - 2975 -> rank 3 ... and so on.
--   2. DENSE_RANK never skips ranks, so tied salaries (e.g., 3000 for
--      SCOTT and FORD) both get rank 2.
--   3. We wrap the ranked result in a subquery and filter for rank = 2.

SELECT *
FROM (
    SELECT EMPNO,
           ENAME,
           SAL,
           DENSE_RANK() OVER (ORDER BY SAL DESC) AS RANK_NUM
    FROM EMP
)
WHERE RANK_NUM = 2;

-- EXPECTED RESULT: SCOTT (3000) and FORD (3000)

In [0]:
-- Find the employee(s) with the MAXIMUM salary in each department.
--
-- Breakdown:
--   1. Inner query uses DENSE_RANK() OVER (PARTITION BY DEPTNO ORDER BY SAL DESC)
--      to rank employees within each department by salary (highest = rank 1).
--   2. DENSE_RANK ensures that if multiple employees share the top salary,
--      they ALL get rank 1.
--   3. The outer query filters for RANK_NUM = 1, i.e. the top earner(s) per dept.

SELECT DEPTNO,
       EMPNO,
       ENAME,
       SAL,
       RANK_NUM
FROM (
    SELECT EMPNO,
           ENAME,
           SAL,
           DEPTNO,
           DENSE_RANK() OVER (PARTITION BY DEPTNO ORDER BY SAL DESC) AS RANK_NUM
    FROM EMP
)
WHERE RANK_NUM = 1
ORDER BY DEPTNO;

-- EXPECTED RESULT:
--   DEPTNO 10 -> KING  (5000)
--   DEPTNO 20 -> SCOTT (3000), FORD (3000)
--   DEPTNO 30 -> BLAKE (2850)

In [0]:
-- Find the SECOND HIGHEST salary PER DEPARTMENT without using DENSE_RANK().
--
-- Breakdown (correlated subquery approach):
--   For each employee E1, the subquery counts how many DISTINCT salaries in
--   the SAME department are strictly GREATER than E1.SAL.
--   If exactly 1 distinct salary is higher, E1 earns the second-highest
--   salary in that department.
--
--   DEPTNO 10: salaries are 5000, 2450, 1300 -> 2nd highest = 2450 (CLARK)
--   DEPTNO 20: salaries are 3000, 3000, 2975, 1100, 800 -> 2nd highest = 2975 (JONES)
--   DEPTNO 30: salaries are 2850, 1600, 1500, 1250, 1250, 950 -> 2nd highest = 1600 (ALLEN)

SELECT *
FROM EMP E1
WHERE 1 = (
    SELECT COUNT(DISTINCT E2.SAL)
    FROM EMP E2
    WHERE E2.DEPTNO = E1.DEPTNO
      AND E2.SAL > E1.SAL
)
ORDER BY DEPTNO;

-- EXPECTED RESULT:
--   DEPTNO 10 -> CLARK  (2450)
--   DEPTNO 20 -> JONES  (2975)
--   DEPTNO 30 -> ALLEN  (1600)